# Required Imports

In [1]:
import os
import re
import glob
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import defaultdict
from tqdm import tqdm

class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super(HardTripletLoss, self).__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()

        # SINTASSI AGGIORNATA PER EVITARE IL DEPRECATION WARNING
        dist.addmm_(inputs, inputs.t(), beta=1, alpha=-2)

        dist = dist.clamp(min=1e-12).sqrt()

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []

        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)

## Parameters

In [2]:
# Set a fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Set the depth and input dimensions (100 packets x 52 subcarriers, Amplitude + Phase)
depth = 32
img_channels = 2  # Raw float32: Channel 0 = Amplitude, Channel 1 = Phase
img_h = 100       # Packets
img_w = 52        # Subcarriers

# Determine the computing device (GPU if available, otherwise CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f'device being used: {device}')

# Configuration settings
n_classes = 40  # Number of identities in the training set
n_epochs = 100  # Number of training epochs
train_batch_size = 32  # Batch size for training
val_batch_size = 1  # Batch size for validation
test_batch_size = 1  # Batch size for testing
learning_rate = 0.0005  # Learning rate for the optimizer
backbone = 'siameseNet_attn'
run_name = f'{backbone}_raw_amp_phase_100p'
models_folder = './output/models_attn_100p/'

device being used: mps


## Load Probe and Gallery sets

In [3]:
def load_raw_pt(pt_pth):
    return torch.load(pt_pth, weights_only=False).float()

def extract_id_from_imagename(filename):
    match = re.search(r'(\d+[mf])\_', filename)
    return match.group(1) if match else None

class FlatReIDTrainDataset(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root
        self.transform = transform
        self.samples, self.id_to_class = self._load_data()

    def _load_data(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            for f in filenames:
                if f.endswith('.pt'):
                    paths.append(os.path.join(dirpath, f))

        unique_ids = sorted(list(set([extract_id_from_imagename(p) for p in paths if extract_id_from_imagename(p)])))
        id_to_class = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}

        samples = []
        for p in paths:
            pid = extract_id_from_imagename(p)
            if pid:
                samples.append((p, id_to_class[pid]))
        return samples, id_to_class

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, target = self.samples[idx]
        img = load_raw_pt(img_path)
        if self.transform:
            img = self.transform(img)
        return img, target

class PKSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, P=8, K=4):
        self.dataset = dataset
        self.P = P
        self.K = K
        self.batch_size = self.P * self.K

        self.id_to_indices = defaultdict(list)
        for idx, (_, target) in enumerate(self.dataset.samples):
            self.id_to_indices[target].append(idx)
        self.unique_ids = list(self.id_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.unique_ids)
        batches = []
        for i in range(0, len(self.unique_ids), self.P):
            p_ids = self.unique_ids[i:i+self.P]
            if len(p_ids) < self.P:
                break
            batch = []
            for pid in p_ids:
                indices = self.id_to_indices[pid]
                if len(indices) >= self.K:
                    batch.extend(random.sample(indices, self.K))
                else:
                    batch.extend(np.random.choice(indices, size=self.K, replace=True))
            batches.append(batch)

        random.shuffle(batches)
        for batch in batches:
            yield from batch

    def __len__(self):
        return (len(self.unique_ids) // self.P) * self.batch_size

class ProbeGalleryValTest(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root
        self.data = self.read_folder()
        self.samples = self.create_samples()
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]
        probe = load_raw_pt(probe_pth)
        gallery = [load_raw_pt(item) for item in gallery_pths]
        if self.transform:
            probe = self.transform(probe)
            gallery = [self.transform(g) for g in gallery]
        return [probe, gallery, is_match, probe_pth, gallery_pths, probe_target, gallery_targets]

    def read_folder(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.endswith('.pt')]
            label = os.path.basename(os.path.normpath(dirpath))
            for item in files:
                paths.append([os.path.join(dirpath, item), label])
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1] == 'Probe']
        gallery = [x[0] for x in self.data if x[1] == 'Gallery']

        samples = []
        for probe in probes:
            probe_id = extract_id_from_imagename(probe)
            if not probe_id:
                continue

            is_match = []
            gallery_targets = []

            for item in gallery:
                item_id = extract_id_from_imagename(item)
                if not item_id:
                    continue

                is_match.append(1 if probe_id == item_id else -1)
                gallery_targets.append(item_id)

            samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

In [4]:
transform = None

train_dataset = FlatReIDTrainDataset('./output/RawTensorData_AmpPhase/D1/', transform=transform)
pk_sampler = PKSampler(train_dataset, P=8, K=4)
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, sampler=pk_sampler, drop_last=True)

test_dataset = ProbeGalleryValTest('./output/RawTensorData_AmpPhase/D2t/', transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test probes: {len(test_dataset)}")

Train samples: 800, Test probes: 410


# Neural Network Models (CNN Raw + Subcarrier Self-Attention)

In [5]:
class SubcarrierAttentionBlock(nn.Module):
    """Multi-Head Self-Attention across the subcarrier/width dimension after spatial pooling."""
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super(SubcarrierAttentionBlock, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # x: [B, C, H', W'] -> pool height to [B, C, W'] -> transpose to sequence [B, W', C]
        seq = x.mean(dim=2).transpose(1, 2)
        norm_seq = self.norm1(seq)
        attn_out, _ = self.attn(norm_seq, norm_seq, norm_seq)
        seq = seq + attn_out
        seq = seq + self.ffn(self.norm2(seq))
        # Return [B, C, 1, W']
        return seq.transpose(1, 2).unsqueeze(2)

class BSN(nn.Module):
    def __init__(self, backbone, img_channels, depth, img_w, img_h, n_classes):
        super(BSN, self).__init__()
        self.backbone = backbone

        self.conv_block = nn.Sequential(
            nn.Conv2d(img_channels, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(depth * 4, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 6))
        self.max_pool = nn.AdaptiveMaxPool2d((1, 6))
        self.subcarrier_attn = SubcarrierAttentionBlock(embed_dim=256, num_heads=4, dropout=0.1)
        output_size = 256 * 6 * 2  # 3072

        self.fc1 = nn.Linear(output_size, 256)
        self.fc2 = nn.Linear(256, 256)
        self.classifier = nn.Linear(256, n_classes)
        self.dropout = nn.Dropout(p=0.3)

    def forward_one(self, x):
        x = self.conv_block(x)
        x_attn = self.subcarrier_attn(x)
        x_mean = self.avg_pool(x_attn).view(x_attn.size(0), -1)
        x_max = self.max_pool(x_attn).view(x_attn.size(0), -1)
        x = torch.cat([x_mean, x_max], dim=1)

        x = self.dropout(F.relu(self.fc1(x)))
        features = self.fc2(x)
        y = self.classifier(features)
        features = F.normalize(features, p=2, dim=1)
        return features, y

    def forward(self, input1, input2):
        output1, output_class1 = self.forward_one(input1)
        output2, output_class2 = self.forward_one(input2)
        return output1, output_class1, output2, output_class2

# Training and Testing functions

In [6]:
def compute_ap(pos_ranks):
    """Calcolo corretto della Average Precision"""
    if len(pos_ranks) == 0:
        return 0.0
    ap = 0.0
    for i, pos in enumerate(pos_ranks):
        ap += (i + 1) / (pos + 1)
    return ap / len(pos_ranks)

def testing_step(model, test_dataloader):
    model.eval()
    test_dataset = test_dataloader.dataset

    if len(test_dataset.samples) == 0:
        print("WARNING: Test dataset is empty! Check ./output/RawTensorData_AmpPhase/D2t/")
        return 0.0, 0.0, 0.0, 0.0

    batch_size = 32

    # 1. Extract Gallery features in batches of 32
    gallery_pths = test_dataset.samples[0][1]
    gallery_ids = [extract_id_from_imagename(p) for p in gallery_pths]
    gallery_features_list = []

    with torch.no_grad():
        for i in range(0, len(gallery_pths), batch_size):
            batch_pths = gallery_pths[i:i+batch_size]
            batch_imgs = torch.stack([load_raw_pt(p) for p in batch_pths]).to(device)
            features, _ = model.forward_one(batch_imgs)
            gallery_features_list.append(features.cpu())
        gallery_features = torch.cat(gallery_features_list, dim=0)

    # 2. Extract Probe features in batches of 32
    probe_pths = [sample[0] for sample in test_dataset.samples]
    probe_ids = [extract_id_from_imagename(p) for p in probe_pths]
    probe_features_list = []

    with torch.no_grad():
        for i in range(0, len(probe_pths), batch_size):
            batch_pths = probe_pths[i:i+batch_size]
            batch_imgs = torch.stack([load_raw_pt(p) for p in batch_pths]).to(device)
            features, _ = model.forward_one(batch_imgs)
            probe_features_list.append(features.cpu())
        probe_features = torch.cat(probe_features_list, dim=0)

    # 3. Vectorized cosine similarity matrix (features are already L2-normalized)
    sim_matrix = torch.matmul(probe_features, gallery_features.t())

    # 4. Compute Rank-K and mAP
    samples = len(probe_ids)
    K = 10
    correct_at_k = np.zeros(K)
    AP_scores = []

    for idx, probe_id in enumerate(probe_ids):
        sorted_indices = torch.argsort(sim_matrix[idx], descending=True).tolist()
        ranked_gallery_ids = [gallery_ids[j] for j in sorted_indices]

        pos_ranks = [rank for rank, gal_id in enumerate(ranked_gallery_ids) if gal_id == probe_id]
        if pos_ranks:
            AP_scores.append(compute_ap(pos_ranks))

        for i in range(K):
            if ranked_gallery_ids[i] == probe_id:
                correct_at_k[i:] += 1
                break

    rank_accs = (correct_at_k / samples) * 100.0
    mAP = np.mean(AP_scores) if AP_scores else 0.0

    print(f'Total samples: {samples}, Rank #1: {rank_accs[0]:.3f}%, Rank #2: {rank_accs[1]:.3f}%, Rank #3: {rank_accs[2]:.3f}%, Rank #5: {rank_accs[4]:.3f}%, Rank #10: {rank_accs[9]:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct_at_k[0]/samples, correct_at_k[4]/samples, correct_at_k[9]/samples, mAP

In [7]:
def training_model(model, train_dataloader, test_dataloader, optimizer, triplet_criterion, n_epochs, scheduler=None):
    os.makedirs(models_folder, exist_ok=True)

    start_epoch = 0
    bestRank1 = 0.0

    # --- CHECKPOINT RESUME LOGIC ---
    existing_checkpoints = glob.glob(f'{models_folder}/best_{run_name}_epoch_*.bin')

    if existing_checkpoints:
        latest_cp = max(existing_checkpoints, key=lambda x: int(re.search(r'epoch_(\d+)\.bin', x).group(1)))
        start_epoch = int(re.search(r'epoch_(\d+)\.bin', latest_cp).group(1)) + 1

        print(f"Resuming training from {latest_cp} at epoch {start_epoch + 1}...")
        model.load_state_dict(torch.load(latest_cp, map_location=device))

        if scheduler is not None:
            for _ in range(start_epoch):
                scheduler.step()

        if os.path.exists(f'{models_folder}/best_{run_name}_state_r1.bin'):
            best_model_temp = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)
            best_model_temp.load_state_dict(torch.load(f'{models_folder}/best_{run_name}_state_r1.bin', map_location=device))
            bestRank1, _, _, _ = testing_step(best_model_temp, test_dataloader)
            print(f"Recovered Best Rank 1: {bestRank1*100:.2f}%")
            del best_model_temp

    # --- TRAINING LOOP ---
    for epoch in range(start_epoch, n_epochs):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            imgs = batch[0].to(device)
            targets = batch[1].to(device)

            features, _ = model.forward_one(imgs)
            loss = triplet_criterion(features, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        if scheduler is not None:
            scheduler.step()

        rank1, rank5, rank10, _ = testing_step(model, test_dataloader)

        if rank1 > bestRank1:
            torch.save(model.state_dict(), f'{models_folder}/best_{run_name}_state_r1.bin')
            bestRank1 = rank1

        for old_cp in glob.glob(f'{models_folder}/best_{run_name}_epoch_*.bin'):
            os.remove(old_cp)

        torch.save(model.state_dict(), f'{models_folder}/best_{run_name}_epoch_{epoch}.bin')

        running_loss = train_loss / len(train_dataloader)
        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {running_loss:.5f}")

## Model Parameters

In [8]:
model = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)
triplet_criterion = HardTripletLoss(margin=0.3)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

## Network Training

In [9]:
# Start training model
training_model(model, train_dataloader, test_dataloader, optimizer, triplet_criterion, n_epochs, scheduler=scheduler)

Total samples: 410, Rank #1: 74.146%, Rank #2: 83.415%, Rank #3: 87.073%, Rank #5: 90.732%, Rank #10: 93.171%, mAP:50.838%
Epoch: [1/100]| Loss: 0.40474


Total samples: 410, Rank #1: 65.610%, Rank #2: 73.415%, Rank #3: 79.756%, Rank #5: 85.366%, Rank #10: 90.488%, mAP:41.853%
Epoch: [2/100]| Loss: 0.41262


Total samples: 410, Rank #1: 62.683%, Rank #2: 70.732%, Rank #3: 76.098%, Rank #5: 82.683%, Rank #10: 89.024%, mAP:43.745%
Epoch: [3/100]| Loss: 0.39552


Total samples: 410, Rank #1: 59.268%, Rank #2: 67.805%, Rank #3: 74.390%, Rank #5: 81.707%, Rank #10: 90.732%, mAP:43.860%
Epoch: [4/100]| Loss: 0.37592


Total samples: 410, Rank #1: 54.390%, Rank #2: 66.829%, Rank #3: 73.902%, Rank #5: 80.488%, Rank #10: 89.024%, mAP:42.592%
Epoch: [5/100]| Loss: 0.36369


Total samples: 410, Rank #1: 52.927%, Rank #2: 64.634%, Rank #3: 72.195%, Rank #5: 81.463%, Rank #10: 87.805%, mAP:40.747%
Epoch: [6/100]| Loss: 0.36342


... (90 outputs left out)

Total samples: 410, Rank #1: 71.463%, Rank #2: 78.293%, Rank #3: 83.415%, Rank #5: 87.805%, Rank #10: 93.415%, mAP:59.813%
Epoch: [97/100]| Loss: 0.08814


Total samples: 410, Rank #1: 72.195%, Rank #2: 77.805%, Rank #3: 83.171%, Rank #5: 87.805%, Rank #10: 93.415%, mAP:59.807%
Epoch: [98/100]| Loss: 0.05706


Total samples: 410, Rank #1: 71.220%, Rank #2: 78.780%, Rank #3: 83.415%, Rank #5: 87.805%, Rank #10: 93.415%, mAP:59.736%
Epoch: [99/100]| Loss: 0.04815


Total samples: 410, Rank #1: 70.976%, Rank #2: 78.780%, Rank #3: 83.415%, Rank #5: 88.049%, Rank #10: 93.659%, mAP:59.751%
Epoch: [100/100]| Loss: 0.06580


## Performance Evaluation

In [10]:
# Initialize the Siamese neural network model with the same architecture and parameters
m1 = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)

# Load pre-trained model weights
m1.load_state_dict(torch.load(f'{models_folder}/best_{run_name}_state_r1.bin', map_location=device))

# Evaluate and print the performance of the model with the respective evaluation metric
print('Model w/ best R1:')
testing_step(m1, test_dataloader)

Model w/ best R1:
Total samples: 410, Rank #1: 78.537%, Rank #2: 84.146%, Rank #3: 88.537%, Rank #5: 93.171%, Rank #10: 96.341%, mAP:59.332%


(np.float64(0.7853658536585366),
 np.float64(0.9317073170731708),
 np.float64(0.9634146341463414),
 np.float64(0.593315978007987))